# External Memory: Remembering Across Sessions & Users

In-context memory is great for a single conversation, but what happens when the script ends? Or when you have 1,000 different users chatting at once? **External Memory** provides the permanent storage and isolation needed for production agents.

---

## 1. The Persistence Problem
Without external storage, every time you run your code, the agent starts with a "blank slate." It has no memory of the history unless you manually provide it every time. External memory bridges this gap by saving conversation state to a database or file system.

## 2. Basic Persistence: File-Based (JSON)
The simplest way to start is by saving your history list to a JSON file. This allows the memory to survive script restarts.

In [6]:
import json
import os
from langchain.schema import HumanMessage, AIMessage, SystemMessage

MEMORY_FILE = "persistent_chat.json"

def save_to_file(messages):
    # Convert message objects to simple dicts for JSON serialization
    serializable = [{"type": m.type, "content": m.content} for m in messages]
    with open(MEMORY_FILE, "w") as f:
        json.dump(serializable, f)

def load_from_file():
    if not os.path.exists(MEMORY_FILE): return []
    with open(MEMORY_FILE, "r") as f:
        data = json.load(f)
        # Re-construct LangChain message objects
        messages = []
        for m in data:
            if m['type'] == 'human': messages.append(HumanMessage(content=m['content']))
            elif m['type'] == 'ai': messages.append(AIMessage(content=m['content']))
        return messages

print("File-based persistence functions ready.")

File-based persistence functions ready.


## 3. Scaling Up: Why Databases?
JSON files are good for one conversation, but if you have multiple users, you would need multiple files. This quickly becomes messy. Databases like **SQLite** (or Postgres/Redis) allow you to store many different conversations in a single, searchable file.

## 4. Managing Parallel Conversations with `thread_id` 
To handle multiple users, we introduce the concept of a **Thread ID**. This is a unique identifier (like a User ID or Session ID) that tells the system exactly which history to retrieve.

### Isolation Logic:
- **User A** has `thread_id="1"` -> retrieves History A.
- **User B** has `thread_id="2"` -> retrieves History B.
- Information never leaks between threads.

## 5. Implementation: Simple Threaded Memory Loop
Let's simulate how a multi-user system works using a simple dictionary lookup.

In [7]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# A dictionary representing our 'Database'
db_mock = {}

def chat_with_thread(user_id, user_input):
    # 1. Retrieve history for this specific thread
    history = db_mock.get(user_id, [])
    
    # 2. Add current query
    history.append(HumanMessage(content=user_input))
    
    # 3. Get AI response
    response = llm.invoke(history)
    
    # 4. Save updated history back to our 'DB'
    history.append(response)
    db_mock[user_id] = history
    
    return response.content

print("Threaded memory simulation ready.")

Threaded memory simulation ready.


## 6. Testing Multi-User Isolation
Let's chat as two different users and verify the AI remembers them independently.

In [8]:
# User 101 says their name is Alice
print(f"User 101: {chat_with_thread('101', 'My name is Alice.')}")

# User 202 says their name is Bob
print(f"User 202: {chat_with_thread('202', 'My name is Bob.')}")

# Ask User 101 who they are
print(f"User 101 Recall: {chat_with_thread('101', 'What is my name?')}")

# Ask User 202 who they are
print(f"User 202 Recall: {chat_with_thread('202', 'What is my name?')}")

User 101: Hello Alice! How can I help you today?
User 202: Alright, Bob.
User 101 Recall: Your name is Alice.
User 202 Recall: Your name is Bob.


## 7. Professional Persistence: LangGraph SQLite Checkpointers
For production, writing your own save/load logic is risky. **LangGraph** provides build-in checkpointers that automatically save every single step of your agent's thought process into a database.

In [9]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# In a real app, this would be 'memory.sqlite'
# We use ':memory:' for a temporary in-memory DB for this demo
memory_db = SqliteSaver.from_conn_string(":memory:")

print("LangGraph SQLite Saver initialized.")

Threaded memory simulation ready.


## 8. Summary & Production Tips
- **Thread ID is Mandatory**: For any web-based agent, you MUST use a session/thread ID to prevent users from seeing each other's data.
- **Sanitization**: If storing data in an external database, be careful of SQL injection or malicious context injected by users.
- **Encryption**: For sensitive data, encrypt your conversation database at rest.
- **Next Step**: Now that we can store history, how do we search through *millions* of messages? That's where **Long-Term Memory (RAG)** comes in.